In [6]:
# main_notebook2_driver.py

"""
Concrete “main” driver for Notebook 2, using the extended utils from notebook2_utils.py.
Loops over:
  • Base scenarios (property + floors + rooms),
  • Time‐of‐day slices (all, day, night),
  • Vendor slices (all vendors, Fibaro),
  • Synthetic anomaly options (no spikes, with spikes).

Saves features only when there is data—skips & logs errors otherwise.
"""

import os
import itertools
import pandas as pd
from utils import log_status, filter_time_of_day, filter_flat_by_floors, aggregate_features, save_features
# ────────────────────────────────────────────────────────────────────────
# 1) LOAD FLATTENED MEASUREMENTS ONCE
# ────────────────────────────────────────────────────────────────────────
log_status("Loading 'flattened_measurements.parquet'…")
try:
    df_flat_full = pd.read_parquet("flattened_measurements.parquet")
    log_status(f"Loaded {len(df_flat_full):,} measurement rows.")
except Exception as e:
    log_status(f"Failed to load flattened_measurements.parquet: {e}", level="ERROR")
    raise
# Quick schema check
missing = {"device","property","value","timestamp","room","floor"} - set(df_flat_full.columns)
if missing:
    log_status(f"Missing expected columns: {missing}", level="WARNING")



[INFO] Loading 'flattened_measurements.parquet'…
[INFO] Loaded 14,883,252 measurement rows.


In [ ]:
# ────────────────────────────────────────────────────────────────────────
# 2) DEFINE BASE SCENARIOS: (property_substrs, floors)
# ────────────────────────────────────────────────────────────────────────

import itertools

log_status("Defining base scenarios…")

# The core sensor types we care about
properties_to_test = ["temp", "humidity", "co2"]

# Generate all non‐empty combinations (single, pairs, triplet)
prop_combinations = []
for r in range(1, len(properties_to_test) + 1):
    prop_combinations.extend(itertools.combinations(properties_to_test, r))

# Floor filters: [] means “all floors”, [n] means floor n
floor_choices = [[], [1], [2], [3], [4], [5], [6], [7]]

# Build the scenario list
base_scenarios = [
    {
        "prop_substrs": list(prop_combo),
        "floors": floor_list
    }
    for prop_combo, floor_list in itertools.product(prop_combinations, floor_choices)
]

log_status(f"Total base scenarios: {len(base_scenarios)}")  # Expect: 7 prop‐combos × 8 floor‐choices = 56


[INFO] Defining base scenarios…
[INFO] Total base scenarios: 56


In [ ]:

# ────────────────────────────────────────────────────────────────────────
# 3) DEFINE EXTENDED FILTER OPTIONS
# ────────────────────────────────────────────────────────────────────────

log_status("Defining time‐of‐day and anomaly options…")

# Time‐of‐day options:
#  • fullday: no time filter
#  • day: hours [8,20)
#  • night: hours [20,8) wrapped
time_options = [
    {"label": "fullday", "func": lambda df: df},
    {"label": "day",      "func": lambda df: filter_time_of_day(df, start_hour=8,  end_hour=20)},
    {"label": "night",    "func": lambda df: filter_time_of_day(df, start_hour=20, end_hour=8)},
]
log_status(f"Time options: {[opt['label'] for opt in time_options]}")




[INFO] Defining time‐of‐day and anomaly options…
[INFO] Time options: ['fullday', 'day', 'night']


In [9]:
# ────────────────────────────────────────────────────────────────────────
# 4) DRIVER LOOP: COMBINE BASE SCENARIOS + TIME SLICES → SAVE FEATURES
# ────────────────────────────────────────────────────────────────────────

MIN_DEVICES = 3
FEATURES_DIR = "features"
os.makedirs(FEATURES_DIR, exist_ok=True)

for base in base_scenarios:
    prop_substrs = base["prop_substrs"]
    floors       = base["floors"]
    base_name    = "_".join(prop_substrs)
    floor_tag    = "allfloors" if not floors else "_".join(map(str, floors))
    scenario_key = f"{base_name}_floor_{floor_tag}"
    log_status(f"Starting scenario: {scenario_key}")

    # 4a) Floor filtering
    df_floor = filter_flat_by_floors(df_flat_full, floors)
    log_status(f"  → {len(df_floor):,} rows after floor filter")
    if df_floor.shape[0] < MIN_DEVICES:
        log_status(f"  • [skip] fewer than {MIN_DEVICES} devices after floor filter", level="WARNING")
        continue

    # 4b) Time‐of‐day slices
    for time_opt in time_options:
        label_time = time_opt["label"]
        df_time    = time_opt["func"](df_floor)
        log_status(f"    Time slice '{label_time}': {len(df_time):,} rows")
        if df_time.shape[0] < MIN_DEVICES:
            log_status(f"    • [skip] fewer than {MIN_DEVICES} devices for time='{label_time}'", level="WARNING")
            continue

        # 4c) Aggregate & save features
        scenario_name = f"{scenario_key}__{label_time}"
        try:
            df_feats = aggregate_features(
                df_flat        = df_time,
                target_substrs = prop_substrs,
                floors         = [],       # already filtered
                rooms_or_zones = []
            )
            ndev = df_feats.shape[0]
            if ndev < MIN_DEVICES:
                log_status(f"    • [skip] only {ndev} devices after aggregation", level="WARNING")
                continue

            out_path = save_features(
                df_feats       = df_feats,
                target_substrs = prop_substrs,
                floors         = floors,
                rooms_or_zones = [],
                out_dir        = FEATURES_DIR,
                custom_name    = scenario_name
            )
            log_status(f"    Saved features for '{scenario_name}' ({ndev} devices) → {out_path}")

        except Exception as e:
            log_status(f"     Failed to aggregate/save '{scenario_name}': {e}", level="ERROR")
            continue


[INFO] Starting scenario: temp_floor_allfloors
[INFO]   → 14,883,252 rows after floor filter
[INFO]     Time slice 'fullday': 14,883,252 rows
[aggregate_features] Dropping 1 devices with no floor mapping:
                                       device room
https://interconnectproject.eu/example/R5_147 None
[INFO]     Saved features for 'temp_floor_allfloors__fullday' (331 devices) → features/temp_floor_allfloors__fullday.pkl
[INFO]     Time slice 'day': 8,438,198 rows
[aggregate_features] Dropping 1 devices with no floor mapping:
                                       device room
https://interconnectproject.eu/example/R5_147 None
[INFO]     Saved features for 'temp_floor_allfloors__day' (327 devices) → features/temp_floor_allfloors__day.pkl
[INFO]     Time slice 'night': 6,445,054 rows
[aggregate_features] Dropping 1 devices with no floor mapping:
                                       device room
https://interconnectproject.eu/example/R5_147 None
[INFO]     Saved features for 'temp_flo

In [10]:
missing_path = "missing_devices.csv"
missing.to_csv(missing_path, index=False)

AttributeError: 'set' object has no attribute 'to_csv'

In [ ]:
# Quick validation of generated feature scenario files
import glob, os
import pandas as pd

FEATURES_DIR = "features"
files = sorted(glob.glob(os.path.join(FEATURES_DIR, "*.*")))
print(f"Found {len(files)} files in '{FEATURES_DIR}':\n")

for path in files:
    name = os.path.basename(path)
    ext  = os.path.splitext(name)[1].lower()
    try:
        if ext in {".pkl", ".pickle"}:
            df = pd.read_pickle(path)
        elif ext in {".csv", ".tsv"}:
            df = pd.read_csv(path, sep=None, engine="python")
        else:
            print(f"  • {name}: unsupported extension, skipping")
            continue
        print(f"  • {name}: {df.shape[0]} devices × {df.shape[1]} features")
        # optional: flag small slices
        if df.shape[0] < 3:
            print(f"      ⚠️ fewer than 3 devices")
    except Exception as e:
        print(f"  • {name}: failed to load ({e})")



Found 159 files in 'features':

  • co2_floor_1__day.pkl: 48 devices × 8 features
  • co2_floor_1__fullday.pkl: 48 devices × 8 features
  • co2_floor_1__night.pkl: 48 devices × 8 features
  • co2_floor_2__day.pkl: 29 devices × 8 features
  • co2_floor_2__fullday.pkl: 29 devices × 8 features
  • co2_floor_2__night.pkl: 29 devices × 8 features
  • co2_floor_3__day.pkl: 52 devices × 8 features
  • co2_floor_3__fullday.pkl: 52 devices × 8 features
  • co2_floor_3__night.pkl: 52 devices × 8 features
  • co2_floor_4__day.pkl: 46 devices × 8 features
  • co2_floor_4__fullday.pkl: 46 devices × 8 features
  • co2_floor_4__night.pkl: 46 devices × 8 features
  • co2_floor_6__day.pkl: 44 devices × 8 features
  • co2_floor_6__fullday.pkl: 44 devices × 8 features
  • co2_floor_6__night.pkl: 44 devices × 8 features
  • co2_floor_7__day.pkl: 5 devices × 8 features
  • co2_floor_7__fullday.pkl: 5 devices × 8 features
  • co2_floor_7__night.pkl: 5 devices × 8 features
  • co2_floor_allfloors__day.pkl: 2

In [2]:
df = pd.read_pickle("features/temp_humidity_co2_floor_allfloors__fullday.pkl")
print(f"Loaded features: {df.shape[0]} devices × {df.shape[1]} features")
print(df.head())

Loaded features: 331 devices × 8 features
                                              device        mean         std  \
0  https://interconnectproject.eu/example/Fibaro_...   24.117214    2.208409   
1        https://interconnectproject.eu/example/R5_1  184.807746  221.761901   
2       https://interconnectproject.eu/example/R5_10  190.230407  232.065034   
3      https://interconnectproject.eu/example/R5_100  177.473842  209.554498   
4      https://interconnectproject.eu/example/R5_101  182.983058  216.865678   

    min     max  count                                               room  \
0  20.3    32.7    639  https://interconnectproject.eu/example/room_ur...   
1  16.6  1239.0  47574  https://interconnectproject.eu/example/zone_VL...   
2  18.0  1506.0  47355  https://interconnectproject.eu/example/zone_VL...   
3  15.9  1667.0  47943  https://interconnectproject.eu/example/roomnam...   
4   0.0  1073.0  47709  https://interconnectproject.eu/example/roomnam...   

   floor  
0  